# Convolutional layer

_Deep Learning_

**A small filter slides over an image, dot-producting as it goes, to spot patterns.**

Images are huge. Connecting every pixel to every neuron would need too many weights.

     A convolutional layer uses a small filter (a tiny grid of weights) that slides across the image.

---

This notebook builds up a convolution one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We create a single convolutional layer and run one image through it. We do it in two steps: (1) define the layer and a dummy image, (2) slide the filters over it and read off the output shape and parameter count.

### Step 1 — Define the conv layer and a test image

A `Conv2d` layer with 1 input channel and 8 output channels learns 8 separate 3x3 filters. With `padding=1`, each filter produces an output the same height and width as the input. We also make one random 28x28 grayscale image shaped `(batch, channels, height, width)` to feed through it.

In [ ]:
import torch
import torch.nn as nn

# 1 input channel (grayscale) -> 8 feature maps, 3x3 filters
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

x = torch.randn(1, 1, 28, 28)        # one 28x28 grayscale image

### Step 2 — Run the image through and inspect the result

Applying the layer slides all 8 filters across the image, producing 8 feature maps stacked into the output. The output keeps the 28x28 spatial size (thanks to padding) but now has 8 channels. The parameter count is `8 * (1*3*3) + 8` — 8 filters of 9 weights each, plus one bias per filter.

In [ ]:
out = conv(x)                        # slide all 8 filters over the image

print("input :", tuple(x.shape))     # (1, 1, 28, 28)
print("output:", tuple(out.shape))   # (1, 8, 28, 28) -- 8 feature maps

param_count = sum(p.numel() for p in conv.parameters())
print("params:", param_count)        # 8*(1*3*3)+8

## Visualize the data & results

_What does a 3x3 vertical-edge filter pull out of a real handwritten digit image?_

### Step 1 — Load a real image and define an edge filter

We grab the first real 8x8 digit (a handwritten 0) and hand-craft a 3x3 filter. This filter has positive weights on its left column and negative on its right, so it responds strongly wherever brightness drops from left to right — a vertical edge.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.images[0].astype(int)      # real 8x8 image of a handwritten 0

kernel = np.array([[1, 0, -1],          # vertical-edge (Sobel-like) filter
                   [1, 0, -1],
                   [1, 0, -1]])

### Step 2 — Slide the filter over the image (a manual convolution)

We compute the convolution by hand: at each position, we line the 3x3 filter up with a 3x3 patch of the image, multiply element-wise, and sum. With no padding, a 3x3 filter over an 8x8 image leaves `8 - 3 + 1 = 6` valid positions in each direction, so the output is 6x6.

In [ ]:
out = np.zeros((6, 6), dtype=int)       # valid conv: 8-3+1 = 6

for i in range(6):
    for j in range(6):
        patch = img[i:i+3, j:j+3]        # the 3x3 window under the filter
        out[i, j] = (patch * kernel).sum()

### Step 3 — Show the image before and after

Side by side, the original digit and the filter's response. In the right panel, red and blue mark where the filter found strong left-to-right brightness changes — the vertical edges of the stroke — while flat regions stay near zero.

In [ ]:
fig, ax = plt.subplots(1, 2)

ax[0].imshow(img, cmap="gray")
ax[0].set_title("Real 8x8 image of a 0")

ax[1].imshow(out, cmap="bwr")
ax[1].set_title("After 3x3 vertical-edge filter")

plt.show()